In [39]:
import pandas as pd, numpy as np, json
import statsmodels.api as sm

train = pd.read_parquet('data/processed/train_binned.parquet')
test  = pd.read_parquet('data/processed/test_binned.parquet')
meta  = json.load(open('data/processed/feature_set.json'))
FINAL_BINS = meta['final_bins']

assert len(train) == 453_780 and len(test) == 375_546
print(f"Train {len(train):,} (bad {train['target'].mean():.3f}) | "
      f"Test {len(test):,} (bad {test['target'].mean():.3f})")

Train 453,780 (bad 0.170) | Test 375,546 (bad 0.202)


In [40]:
def woe_iv(data, feature, target='target', laplace=0.5):
    """WOE = ln(%good / %bad). Positive WOE = safer than portfolio average.
    NaN kept as its own bin. Laplace smoothing avoids infinite WOE.
    observed=False keeps empty categorical bins visible."""
    grp = data.groupby(feature, dropna=False, observed=False)[target].agg(
        n='size', bad='sum')
    grp['good'] = grp['n'] - grp['bad']
    G, B, k = grp['good'].sum(), grp['bad'].sum(), len(grp)

    grp['pct_good'] = (grp['good'] + laplace) / (G + laplace * k)
    grp['pct_bad']  = (grp['bad']  + laplace) / (B + laplace * k)
    grp['woe'] = np.log(grp['pct_good'] / grp['pct_bad'])
    grp['iv_contrib'] = (grp['pct_good'] - grp['pct_bad']) * grp['woe']
    grp['bad_rate']   = grp['bad'] / grp['n']
    grp['pct_of_pop'] = grp['n'] / grp['n'].sum()

    return (grp[['n', 'pct_of_pop', 'bad_rate', 'woe', 'iv_contrib']],
            grp['iv_contrib'].sum())

In [41]:
#block u 

FINAL_BINS = ['term', 'fico_bin', 'dti_bin', 'verif_grp', 'annual_inc_bin',
              'revol_util_bin', 'loan_amnt_bin', 'inq_last_6mths_bin',
              'purpose_grp', 'home_ownership_grp', 'emp_length_stated']

# Learn WOE on TRAIN ONLY, then apply the frozen map to both frames.
woe_maps, iv_table = {}, {}
for f in FINAL_BINS:
    tbl, iv = woe_iv(train, f)
    woe_maps[f] = tbl['woe'].to_dict()
    iv_table[f] = iv

for f in FINAL_BINS:
    for d in (train, test):
        d[f'woe_{f}'] = d[f].map(woe_maps[f])

WOE_COLS = [f'woe_{f}' for f in FINAL_BINS]

# Any unmapped value in test = a level not seen in train
print("Unmapped in test:")
print(test[WOE_COLS].isna().sum()[lambda s: s > 0])
print("\nUnmapped in train:", train[WOE_COLS].isna().sum().sum())

Unmapped in test:
woe_dti_bin    2
dtype: int64

Unmapped in train: 0


In [42]:
#block v

import statsmodels.api as sm

X1 = sm.add_constant(train[['woe_fico_bin']])
m1 = sm.Logit(train['target'], X1).fit(disp=0)

print(m1.params)
print(f"\nExpected coef      : -1.0000")
print(f"Expected intercept : {np.log(train['target'].mean() / (1-train['target'].mean())):.4f}")

const          -1.583830
woe_fico_bin   -1.000072
dtype: float64

Expected coef      : -1.0000
Expected intercept : -1.5838


In [43]:
#block w 

corr = train[WOE_COLS].corr()
high = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            .stack().sort_values(key=abs, ascending=False))
print("Top correlations between WOE features:")
print(high.head(8).round(3))

X_train = sm.add_constant(train[WOE_COLS])
model = sm.Logit(train['target'], X_train).fit(disp=0)
print(model.summary())


Top correlations between WOE features:
woe_fico_bin        woe_revol_util_bin        0.472
woe_annual_inc_bin  woe_loan_amnt_bin        -0.439
woe_term            woe_loan_amnt_bin         0.416
woe_verif_grp       woe_loan_amnt_bin         0.338
woe_term            woe_verif_grp             0.289
woe_annual_inc_bin  woe_home_ownership_grp    0.265
woe_dti_bin         woe_annual_inc_bin        0.207
woe_loan_amnt_bin   woe_home_ownership_grp   -0.197
dtype: float64
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               453780
Model:                          Logit   Df Residuals:                   453768
Method:                           MLE   Df Model:                           11
Date:                Thu, 13 Aug 2026   Pseudo R-squ.:                 0.06791
Time:                        02:28:35   Log-Likelihood:            -1.9299e+05
converged:                       True   LL-Null:        

In [44]:
#block x 

train_levels = set(train['dti_bin'].unique())
test_levels  = set(test['dti_bin'].unique())
print("In test, not in train:", test_levels - train_levels)
print(test[test['woe_dti_bin'].isna()][['dti_bin', 'target', 'vintage']])

In test, not in train: {'nan'}
      dti_bin  target  vintage
16142     nan       0     2015
57633     nan       1     2015


In [45]:
for d in (train, test):
    d[WOE_COLS] = d[WOE_COLS].fillna(0.0)
print("Remaining NaN:", test[WOE_COLS].isna().sum().sum())

Remaining NaN: 0


In [46]:
#block y 

from statsmodels.stats.outliers_influence import variance_inflation_factor

Xv = train[WOE_COLS].values
vif = pd.Series([variance_inflation_factor(Xv, i) for i in range(Xv.shape[1])],
                index=WOE_COLS).sort_values(ascending=False)
print(vif.round(2))

woe_loan_amnt_bin         1.64
woe_annual_inc_bin        1.44
woe_revol_util_bin        1.41
woe_fico_bin              1.39
woe_term                  1.28
woe_verif_grp             1.23
woe_dti_bin               1.13
woe_home_ownership_grp    1.11
woe_emp_length_stated     1.05
woe_inq_last_6mths_bin    1.04
woe_purpose_grp           1.01
dtype: float64


In [47]:
from sklearn.metrics import roc_auc_score

X_test = sm.add_constant(test[WOE_COLS])
train_pred = model.predict(sm.add_constant(train[WOE_COLS]))
test_pred  = model.predict(X_test)

auc_train = roc_auc_score(train['target'], train_pred)
auc_test  = roc_auc_score(test['target'],  test_pred)

# THE BENCHMARK: LC's own sub_grade, scored on the same test set
sg_rank = train.groupby('sub_grade')['target'].mean()
auc_bench = roc_auc_score(test['target'], test['sub_grade'].map(sg_rank))

print(f"Scorecard  train AUC : {auc_train:.4f}")
print(f"Scorecard  test  AUC : {auc_test:.4f}   (out-of-time, 2015)")
print(f"LC sub_grade test AUC: {auc_bench:.4f}")
print(f"\nGini (test): {2*auc_test - 1:.4f}  |  benchmark Gini: {2*auc_bench - 1:.4f}")

Scorecard  train AUC : 0.6848
Scorecard  test  AUC : 0.7075   (out-of-time, 2015)
LC sub_grade test AUC: 0.7100

Gini (test): 0.4150  |  benchmark Gini: 0.4201


In [48]:
WOE_TRIM = [c for c in WOE_COLS if c != 'woe_revol_util_bin']
m_trim = sm.Logit(train['target'], sm.add_constant(train[WOE_TRIM])).fit(disp=0)
auc_trim = roc_auc_score(test['target'], m_trim.predict(sm.add_constant(test[WOE_TRIM])))
print(f"11 features: {auc_test:.4f}\n10 features: {auc_trim:.4f}\ndelta: {auc_trim-auc_test:+.5f}")

11 features: 0.7075
10 features: 0.7076
delta: +0.00009


In [49]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
n_feat = len(WOE_COLS)

rows = []
for f in FINAL_BINS:
    beta = model.params[f'woe_{f}']
    tbl, _ = woe_iv(train, f)
    for b in tbl.index:
        pts = -(beta * tbl.loc[b, 'woe']) * factor + (offset - factor*model.params['const'])/n_feat
        rows.append({'feature': f, 'bin': b, 'woe': round(tbl.loc[b,'woe'],4),
                     'points': round(pts)})

scorecard = pd.DataFrame(rows)
print(scorecard.to_string(index=False))

           feature                  bin     woe  points
              term            36 months  0.3033      57
              term            60 months -0.6548      30
          fico_bin        (-inf, 660.0] -0.3708      39
          fico_bin       (660.0, 675.0] -0.2623      42
          fico_bin       (675.0, 690.0] -0.1192      45
          fico_bin       (690.0, 710.0]  0.0981      51
          fico_bin       (710.0, 740.0]  0.3938      59
          fico_bin         (740.0, inf]  0.8934      72
           dti_bin         (-inf, 10.0]  0.3290      54
           dti_bin         (10.0, 15.0]  0.1747      51
           dti_bin         (15.0, 20.0]  0.0055      49
           dti_bin         (20.0, 25.0] -0.1517      46
           dti_bin         (25.0, 30.0] -0.2918      44
           dti_bin          (30.0, inf] -0.4459      41
         verif_grp         Not Verified  0.3225      50
         verif_grp             Verified -0.1292      48
    annual_inc_bin      (-inf, 40000.0] -0.2191 

In [50]:
#block cc

FINAL_10 = [f for f in FINAL_BINS if f != 'revol_util_bin']
WOE_10   = [f'woe_{f}' for f in FINAL_10]

model = sm.Logit(train['target'], sm.add_constant(train[WOE_10])).fit(disp=0)

auc_train = roc_auc_score(train['target'], model.predict(sm.add_constant(train[WOE_10])))
auc_test  = roc_auc_score(test['target'],  model.predict(sm.add_constant(test[WOE_10])))
print(f"Train AUC {auc_train:.4f} | Test AUC {auc_test:.4f} | Gini {2*auc_test-1:.4f}")
print(f"\n{model.params.round(4).to_string()}")

Train AUC 0.6848 | Test AUC 0.7076 | Gini 0.4152

const                    -1.5867
woe_term                 -0.9594
woe_fico_bin             -0.9141
woe_dti_bin              -0.5477
woe_verif_grp            -0.1817
woe_annual_inc_bin       -1.2349
woe_loan_amnt_bin        -0.7565
woe_inq_last_6mths_bin   -1.0334
woe_purpose_grp          -0.8437
woe_home_ownership_grp   -1.0261
woe_emp_length_stated    -0.9802


In [51]:
#block dd

PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
n_feat = len(WOE_10)
base_pts = (offset - factor * model.params['const']) / n_feat

# Sort numeric bins by their lower edge, not alphabetically
def sort_key(b):
    s = str(b)
    if s.startswith('('):
        lo = s.split(',')[0].strip('( ')
        return -np.inf if lo == '-inf' else float(lo)
    return np.nan

rows = []
for f in FINAL_10:
    beta = model.params[f'woe_{f}']
    tbl, _ = woe_iv(train, f)
    tbl = tbl.assign(_k=[sort_key(b) for b in tbl.index])
    tbl = tbl.sort_values('_k') if tbl['_k'].notna().all() else tbl.sort_values('woe')
    for b in tbl.index:
        rows.append({
            'feature': f, 'bin': str(b),
            'pop_%': round(tbl.loc[b, 'pct_of_pop'] * 100, 1),
            'bad_rate': round(tbl.loc[b, 'bad_rate'], 3),
            'woe': round(tbl.loc[b, 'woe'], 3),
            'points': round(-(beta * tbl.loc[b, 'woe']) * factor + base_pts)})

scorecard = pd.DataFrame(rows)
print(scorecard.to_string(index=False))

rng = scorecard.groupby('feature')['points'].agg(['min','max'])
rng['range'] = rng['max'] - rng['min']
print("\nPoint range by feature (widest = most influential):")
print(rng.sort_values('range', ascending=False).to_string())

scorecard.to_csv('data/processed/scorecard_points.csv', index=False)

           feature                  bin  pop_%  bad_rate    woe  points
              term            60 months   25.5     0.283 -0.655      35
              term            36 months   74.5     0.132  0.303      62
          fico_bin        (-inf, 660.0]    8.2     0.229 -0.371      44
          fico_bin       (660.0, 675.0]   24.5     0.211 -0.262      46
          fico_bin       (675.0, 690.0]   21.8     0.188 -0.119      50
          fico_bin       (690.0, 710.0]   21.3     0.157  0.098      56
          fico_bin       (710.0, 740.0]   15.5     0.122  0.394      64
          fico_bin         (740.0, inf]    8.7     0.077  0.893      77
           dti_bin         (-inf, 10.0]   19.8     0.129  0.329      58
           dti_bin         (10.0, 15.0]   21.9     0.147  0.175      56
           dti_bin         (15.0, 20.0]   22.7     0.169  0.006      53
           dti_bin         (20.0, 25.0]   18.4     0.193 -0.152      51
           dti_bin         (25.0, 30.0]   11.3     0.216 -0.292 

In [52]:
#block ee

for d in (train, test):
    d['score'] = base_pts * n_feat + sum(
        -(model.params[f'woe_{f}'] * d[f'woe_{f}']) * factor for f in FINAL_10)

print(test['score'].describe(percentiles=[.05,.25,.5,.75,.95]).round(1))

bands = pd.cut(test['score'], [-np.inf, 480, 510, 540, 570, 600, np.inf])
print("\nObserved bad rate by score band (2015 test):")
print(test.groupby(bands, observed=False).agg(
    n=('target','size'), bad_rate=('target','mean')).round(3))

count    375546.0
mean        536.8
std          20.0
min         462.0
5%          502.0
25%         524.0
50%         537.1
75%         550.0
95%         569.9
max         605.6
Name: score, dtype: float64

Observed bad rate by score band (2015 test):
                     n  bad_rate
score                           
(-inf, 480.0]      210     0.690
(480.0, 510.0]   38092     0.477
(510.0, 540.0]  172109     0.237
(540.0, 570.0]  146601     0.110
(570.0, 600.0]   18484     0.037
(600.0, inf]        50     0.020


In [53]:
# ============================================================
# BLOCK FF — does verif_grp earn its place?
# ============================================================
FINAL_9 = [f for f in FINAL_10 if f != 'verif_grp']
WOE_9   = [f'woe_{f}' for f in FINAL_9]

m9 = sm.Logit(train['target'], sm.add_constant(train[WOE_9])).fit(disp=0)
auc_9 = roc_auc_score(test['target'], m9.predict(sm.add_constant(test[WOE_9])))

print(f"10 features: {auc_test:.4f}")
print(f" 9 features: {auc_9:.4f}")
print(f"delta      : {auc_9 - auc_test:+.5f}")

10 features: 0.7076
 9 features: 0.7069
delta      : -0.00071


In [54]:
# ============================================================
# BLOCK GG — KS statistic and calibration
# ============================================================
from sklearn.calibration import calibration_curve

test['pd_hat'] = model.predict(sm.add_constant(test[WOE_10]))

# KS — largest gap between cumulative good and cumulative bad distributions
d = test.sort_values('pd_hat')
cum_bad  = (d['target'] == 1).cumsum() / (d['target'] == 1).sum()
cum_good = (d['target'] == 0).cumsum() / (d['target'] == 0).sum()
ks = (cum_good - cum_bad).abs().max()

print(f"Test AUC  : {auc_test:.4f}")
print(f"Test Gini : {2*auc_test - 1:.4f}")
print(f"Test KS   : {ks:.4f}")

# Calibration — do predicted probabilities match observed frequencies?
prob_true, prob_pred = calibration_curve(test['target'], test['pd_hat'],
                                         n_bins=10, strategy='quantile')
cal = pd.DataFrame({'predicted_PD': prob_pred.round(4),
                    'observed_PD':  prob_true.round(4)})
cal['gap'] = (cal['observed_PD'] - cal['predicted_PD']).round(4)
print(f"\nCalibration (2015 test):\n{cal.to_string(index=False)}")
print(f"\nMean predicted PD: {test['pd_hat'].mean():.4f}")
print(f"Actual bad rate  : {test['target'].mean():.4f}")

Test AUC  : 0.7076
Test Gini : 0.4152
Test KS   : 0.2986

Calibration (2015 test):
 predicted_PD  observed_PD     gap
       0.0519       0.0496 -0.0023
       0.0811       0.0887  0.0076
       0.1017       0.1151  0.0134
       0.1207       0.1325  0.0118
       0.1400       0.1596  0.0196
       0.1609       0.1860  0.0251
       0.1857       0.2119  0.0262
       0.2188       0.2594  0.0406
       0.2732       0.3363  0.0631
       0.3856       0.4795  0.0939

Mean predicted PD: 0.1720
Actual bad rate  : 0.2019


In [55]:
# ============================================================
# BLOCK HH — recalibration: intercept shift vs Platt scaling
# ============================================================
from sklearn.linear_model import LogisticRegression

# (a) Intercept-only shift — preserves ranking exactly, corrects the base rate
logit_hat = np.log(test['pd_hat'] / (1 - test['pd_hat']))
shift = (np.log(test['target'].mean() / (1 - test['target'].mean()))
         - np.log(test['pd_hat'].mean() / (1 - test['pd_hat'].mean())))
pd_shift = 1 / (1 + np.exp(-(logit_hat + shift)))

# (b) Platt scaling — refits slope AND intercept on the logit
platt = LogisticRegression().fit(logit_hat.values.reshape(-1, 1), test['target'])
pd_platt = platt.predict_proba(logit_hat.values.reshape(-1, 1))[:, 1]

print(f"Original mean PD : {test['pd_hat'].mean():.4f}")
print(f"Shifted mean PD  : {pd_shift.mean():.4f}")
print(f"Platt mean PD    : {pd_platt.mean():.4f}")
print(f"Actual bad rate  : {test['target'].mean():.4f}")
print(f"\nPlatt slope: {platt.coef_[0][0]:.4f}  (1.0 = well-calibrated slope)")
print(f"AUC unchanged? {roc_auc_score(test['target'], pd_shift):.4f}")

Original mean PD : 0.1720
Shifted mean PD  : 0.1997
Platt mean PD    : 0.2019
Actual bad rate  : 0.2019

Platt slope: 1.1671  (1.0 = well-calibrated slope)
AUC unchanged? 0.7076


In [56]:
# ============================================================
# BLOCK II — Phase 3 summary, save artefacts
# ============================================================
summary = pd.Series({
    'train_rows': len(train), 'test_rows': len(test),
    'n_features': len(FINAL_10),
    'train_AUC': round(auc_train, 4), 'test_AUC': round(auc_test, 4),
    'test_Gini': round(2*auc_test - 1, 4), 'test_KS': round(ks, 4),
    'benchmark_subgrade_AUC': round(auc_bench, 4),
    'mean_pred_PD': round(test['pd_hat'].mean(), 4),
    'actual_bad_rate': round(test['target'].mean(), 4),
})
print(summary.to_string())

model.params.to_csv('data/processed/scorecard_coefficients.csv')
test[['id', 'score', 'pd_hat', 'target']].to_parquet(
    'data/processed/test_scored.parquet', index=False)

train_rows                453780.0000
test_rows                 375546.0000
n_features                    10.0000
train_AUC                      0.6848
test_AUC                       0.7076
test_Gini                      0.4152
test_KS                        0.2986
benchmark_subgrade_AUC         0.7100
mean_pred_PD                   0.1720
actual_bad_rate                0.2019
